# Fine-tuning Models on SageMaker

This notebook demonstrates how to fine-tune a pre-trained model on Amazon SageMaker to improve its performance on a specific task. Fine-tuning is a powerful technique that allows you to adapt a pre-trained model to your specific use case, often resulting in significant performance improvements.

## What is Fine-tuning?

Fine-tuning is the process of taking a model that has been pre-trained on a large dataset and then further training it on a smaller, task-specific dataset. This allows the model to adapt its learned features to the specific characteristics of your task.

Benefits of fine-tuning include:
- Improved accuracy on domain-specific tasks
- Faster training compared to training from scratch
- Better generalization with smaller datasets
- Adaptation to specific language patterns or terminology

## How Fine-tuning Complements Other Optimization Techniques

In previous notebooks, we explored various model optimization techniques:
- **Quantization**: Reducing the precision of model weights
- **Pruning**: Removing unnecessary connections in the model
- **Knowledge Distillation**: Creating smaller student models that learn from larger teacher models

Fine-tuning complements these techniques by:
1. Improving model quality before applying other optimizations
2. Recovering performance that might be lost during optimization
3. Adapting optimized models to specific domains

## What We'll Cover

In this notebook, we will:
1. Evaluate a pre-trained model on a sentiment analysis task
2. Prepare a dataset for fine-tuning
3. Configure and run a distributed fine-tuning job on SageMaker using GPU instances
4. Deploy and evaluate the fine-tuned model
5. Compare performance before and after fine-tuning

Let's get started!

## Setup

First, let's install the specific packages needed for this fine-tuning notebook.

In [ ]:
# Install only the packages needed for this notebook
!pip install -q "torch==1.13.1" "transformers==4.26.0" "datasets==2.10.1" "boto3>=1.26.0" "sagemaker>=2.130.0" "pandas>=1.5.0" "numpy>=1.23.0" "matplotlib>=3.6.0" "scikit-learn>=1.2.0"

Now, let's import the necessary libraries and set up our SageMaker environment.

In [ ]:
# Import only what we need for this notebook
import os
import json
import time
from datetime import datetime

# Data processing
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Visualization
import matplotlib.pyplot as plt

# Machine learning
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    pipeline
)
from datasets import load_dataset

# AWS
import boto3
import sagemaker
from sagemaker.huggingface import HuggingFace
from sagemaker.huggingface.model import HuggingFaceModel

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load workshop configuration if available
try:
    with open('workshop_config.json', 'r') as f:
        workshop_config = json.load(f)
    
    # Use configuration values
    base_model = workshop_config.get("base_model", "distilbert-base-uncased-finetuned-sst-2-english")
    task = workshop_config.get("task", "sequence-classification")
except FileNotFoundError:
    # Default values if config not found
    base_model = "distilbert-base-uncased-finetuned-sst-2-english"
    task = "sequence-classification"
    print("Workshop configuration not found. Using default values.")

In [ ]:
# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.session.Session().region_name
bucket = sagemaker_session.default_bucket()
prefix = "fine-tuning-workshop"

print(f"SageMaker session established in region: {region}")
print(f"Using S3 bucket: {bucket}")
print(f"Using S3 prefix: {prefix}")

## 1. Evaluate the Pre-trained Model

Before fine-tuning, let's evaluate the pre-trained model on our target task to establish a baseline.

In [ ]:
# Load the pre-trained model and tokenizer
print(f"Loading model: {base_model}")
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForSequenceClassification.from_pretrained(base_model)

# Create a sentiment analysis pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Test with a few examples
test_texts = [
    "I love this product! It's amazing and works perfectly.",
    "This is terrible. Completely disappointed with the quality.",
    "It's okay, not great but not bad either.",
    "The customer service was excellent but the product was mediocre."
]

# Run inference
results = sentiment_pipeline(test_texts)

# Display results
for text, result in zip(test_texts, results):
    print(f"Text: {text}")
    print(f"Sentiment: {result['label']} (Score: {result['score']:.4f})\n")

## 2. Prepare Dataset for Fine-tuning

Now, let's prepare a dataset for fine-tuning. We'll use the Amazon Reviews Polarity dataset, which contains product reviews labeled as positive or negative.

In [ ]:
# Load the dataset with error handling for SageMaker compatibility
try:
    from datasets import load_dataset
    dataset = load_dataset("amazon_polarity")
    print(f"Dataset loaded: {dataset}")
except Exception as e:
    print(f"Error loading amazon_polarity dataset: {e}")
    print("Falling back to IMDB dataset...")
    dataset = load_dataset("imdb")
    print(f"IMDB dataset loaded: {dataset}")

# Display dataset information
print(f"Train set size: {len(dataset['train'])}")
print(f"Test set size: {len(dataset['test'])}")

# Display a few examples
print("\nSample examples:")
for i in range(3):
    example = dataset['train'][i]
    
    # Handle different dataset structures (amazon_polarity vs imdb)
    if 'label' in example:
        label = example['label']
        label_desc = "(0=negative, 1=positive)"
    else:
        label = example.get('sentiment', 'unknown')
        label_desc = ""
        
    if 'title' in example and 'content' in example:
        title = example['title']
        content = example['content'][:100]
    else:
        title = "N/A"
        content = example.get('text', '')[:100]
        
    print(f"Label: {label} {label_desc}")
    print(f"Title: {title}")
    print(f"Content: {content}...\n")

In [ ]:
# Since the dataset is large, let's use a smaller subset for fine-tuning
train_dataset = dataset["train"].shuffle(seed=42).select(range(10000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

print(f"Reduced train set size: {len(train_dataset)}")
print(f"Reduced test set size: {len(test_dataset)}")

In [ ]:
# Prepare the dataset for the model
def tokenize_function(examples):
    # Handle different dataset structures (amazon_polarity vs imdb)
    if 'title' in examples and 'content' in examples:
        # Amazon Polarity dataset
        texts = [title + ". " + content for title, content in zip(examples["title"], examples["content"])]
    else:
        # IMDB dataset
        texts = examples.get("text", [])
    
    return tokenizer(texts, padding="max_length", truncation=True, max_length=128)

# Tokenize the datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Set the format for PyTorch
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Datasets prepared for fine-tuning")

## 3. Create Training Script

Now, let's create a training script that will be used by SageMaker for fine-tuning. This script will handle the distributed training process.

In [ ]:
# The training script is already created at scripts/train_sentiment.py
# Let's verify it exists
!ls -la scripts/train_sentiment.py

## 4. Save Datasets to S3

Now, let's save our prepared datasets to S3 so they can be accessed by the SageMaker training job.

In [ ]:
# Create directories for datasets
os.makedirs("data/train", exist_ok=True)
os.makedirs("data/test", exist_ok=True)

# Save datasets locally
tokenized_train.save_to_disk("data/train")
tokenized_test.save_to_disk("data/test")

# Upload datasets to S3
train_s3_uri = sagemaker_session.upload_data(path="data/train", bucket=bucket, key_prefix=f"{prefix}/data/train")
test_s3_uri = sagemaker_session.upload_data(path="data/test", bucket=bucket, key_prefix=f"{prefix}/data/test")

print(f"Train dataset uploaded to: {train_s3_uri}")
print(f"Test dataset uploaded to: {test_s3_uri}")

## 5. Configure and Launch Distributed Fine-tuning Job

Now, let's configure and launch a distributed fine-tuning job on SageMaker using GPU instances.

In [ ]:
# Define hyperparameters
hyperparameters = {
    "epochs": 3,
    "train-batch-size": 32,
    "eval-batch-size": 64,
    "learning-rate": 5e-5,
    "warmup-steps": 500,
    "model-id": base_model,
    "fp16": True
}

# Define metric definitions for CloudWatch monitoring
metric_definitions = [
    {"Name": "train_loss", "Regex": "train_loss: ([0-9\\.]+)"},
    {"Name": "eval_loss", "Regex": "eval_loss: ([0-9\\.]+)"},
    {"Name": "eval_accuracy", "Regex": "eval_accuracy: ([0-9\\.]+)"},
    {"Name": "eval_f1", "Regex": "eval_f1: ([0-9\\.]+)"},
    {"Name": "eval_precision", "Regex": "eval_precision: ([0-9\\.]+)"},
    {"Name": "eval_recall", "Regex": "eval_recall: ([0-9\\.]+)"}
]

In [ ]:
# Configure the Hugging Face estimator for distributed training
huggingface_estimator = HuggingFace(
    entry_point="train_sentiment.py",
    source_dir="scripts",
    role=role,
    instance_count=2,  # Use 2 instances for distributed training
    instance_type="ml.g4dn.xlarge",  # GPU instance type
    transformers_version="4.26.0",
    pytorch_version="1.13.1",
    py_version="py39",
    hyperparameters=hyperparameters,
    metric_definitions=metric_definitions,
    distribution={
        "pytorchddp": {
            "enabled": True  # Enable PyTorch Distributed Data Parallel
        }
    },
    max_run=7200,  # Maximum runtime in seconds (2 hours)
    output_path=f"s3://{bucket}/{prefix}/output"
)

# Define the data channels
data_channels = {
    "train": train_s3_uri,
    "test": test_s3_uri
}

In [ ]:
# Launch the training job
job_name = f"fine-tuning-{time.strftime('%Y-%m-%d-%H-%M-%S')}"
print(f"Launching training job: {job_name}")

huggingface_estimator.fit(data_channels, job_name=job_name)

print(f"Training job completed: {job_name}")

## 6. Deploy the Fine-tuned Model

Now that we have fine-tuned our model, let's deploy it to a SageMaker endpoint for inference.

In [ ]:
# Create a Hugging Face model from the trained artifacts
fine_tuned_model = HuggingFaceModel(
    model_data=huggingface_estimator.model_data,
    role=role,
    transformers_version="4.26.0",
    pytorch_version="1.13.1",
    py_version="py39"
)

# Deploy the model to an endpoint
endpoint_name = f"fine-tuned-sentiment-{time.strftime('%Y-%m-%d-%H-%M-%S')}"
print(f"Deploying model to endpoint: {endpoint_name}")

predictor = fine_tuned_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g4dn.xlarge",  # GPU instance for inference
    endpoint_name=endpoint_name
)

print(f"Model deployed to endpoint: {endpoint_name}")

## 7. Evaluate the Fine-tuned Model

Let's evaluate our fine-tuned model on the same examples we used earlier to see the improvement.

In [ ]:
# Prepare the test examples
test_payload = {
    "inputs": test_texts
}

# Get predictions from the endpoint
response = predictor.predict(test_payload)

# Display results
print("Fine-tuned model predictions:")
for text, result in zip(test_texts, response):
    label = "POSITIVE" if result["label"] == "LABEL_1" else "NEGATIVE"
    score = result["score"]
    print(f"Text: {text}")
    print(f"Sentiment: {label} (Score: {score:.4f})\n")

## 8. Compare Performance

Let's compare the performance of the pre-trained model and the fine-tuned model on a larger test set.

In [ ]:
# Prepare a larger test set
evaluation_texts = []
evaluation_labels = []

# Get 100 examples from the test dataset
for i in range(100):
    example = test_dataset[i]
    text = example["title"] + ". " + example["content"]
    label = example["label"]
    evaluation_texts.append(text)
    evaluation_labels.append(label)

# Get predictions from the pre-trained model
pretrained_results = sentiment_pipeline(evaluation_texts)
pretrained_predictions = [1 if result["label"] == "POSITIVE" else 0 for result in pretrained_results]

# Get predictions from the fine-tuned model
finetuned_payload = {"inputs": evaluation_texts}
finetuned_results = predictor.predict(finetuned_payload)
finetuned_predictions = [1 if result["label"] == "LABEL_1" else 0 for result in finetuned_results]

# Calculate accuracy
pretrained_accuracy = accuracy_score(evaluation_labels, pretrained_predictions)
finetuned_accuracy = accuracy_score(evaluation_labels, finetuned_predictions)

# Calculate precision, recall, and F1 score
pretrained_precision, pretrained_recall, pretrained_f1, _ = precision_recall_fscore_support(
    evaluation_labels, pretrained_predictions, average="binary"
)
finetuned_precision, finetuned_recall, finetuned_f1, _ = precision_recall_fscore_support(
    evaluation_labels, finetuned_predictions, average="binary"
)

# Display results
print("Performance Comparison:")
print(f"Pre-trained model accuracy: {pretrained_accuracy:.4f}")
print(f"Fine-tuned model accuracy: {finetuned_accuracy:.4f}")
print(f"Accuracy improvement: {(finetuned_accuracy - pretrained_accuracy) * 100:.2f}%\n")

print(f"Pre-trained model F1 score: {pretrained_f1:.4f}")
print(f"Fine-tuned model F1 score: {finetuned_f1:.4f}")
print(f"F1 score improvement: {(finetuned_f1 - pretrained_f1) * 100:.2f}%")

In [ ]:
# Visualize the results
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
pretrained_values = [pretrained_accuracy, pretrained_precision, pretrained_recall, pretrained_f1]
finetuned_values = [finetuned_accuracy, finetuned_precision, finetuned_recall, finetuned_f1]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, pretrained_values, width, label='Pre-trained Model')
rects2 = ax.bar(x + width/2, finetuned_values, width, label='Fine-tuned Model')

ax.set_ylabel('Score')
ax.set_title('Performance Comparison: Pre-trained vs. Fine-tuned Model')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)

# Add value labels on bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

fig.tight_layout()
plt.show()

## 9. Clean Up Resources

Finally, let's clean up the resources we created to avoid incurring unnecessary costs.

In [ ]:
# Delete the endpoint
print(f"Deleting endpoint: {endpoint_name}")
predictor.delete_endpoint()
print("Endpoint deleted")

## Conclusion

In this notebook, we've demonstrated how to fine-tune a pre-trained model on Amazon SageMaker using distributed training on GPU instances. We've seen how fine-tuning can significantly improve model performance on a specific task.

Key takeaways:
1. Fine-tuning adapts pre-trained models to specific domains and tasks
2. Distributed training on GPU instances accelerates the fine-tuning process
3. Fine-tuned models often show significant performance improvements over pre-trained models
4. SageMaker provides a scalable platform for fine-tuning and deploying models

In the next notebook, we'll explore cost analysis to understand the financial implications of different model optimization techniques.